In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

file_path = "/content/drive/MyDrive/AB_NYC_2019.csv"

print("File exists:", os.path.exists(file_path))

File exists: True


In [3]:
import pandas as pd
import numpy as np
import re
import time
import os

df = pd.read_csv(
    "/content/drive/MyDrive/AB_NYC_2019.csv",
    dtype={
        "id": "int64",
        "name": "string",
        "host_id": "int64",
        "host_name": "string",
        "neighbourhood_group": "string",
        "neighbourhood": "string",
        "latitude": "float64",
        "longitude": "float64",
        "room_type": "string",
        "price": "int64",
        "minimum_nights": "int64",
        "number_of_reviews": "int64",
        "reviews_per_month": "float64",
        "calculated_host_listings_count": "int64",
        "availability_365": "int64"
    },
    parse_dates=["last_review"]
)

print("Shape:", df.shape)
print(df.head())

Shape: (48895, 16)
     id                                              name  host_id  \
0  2539                Clean & quiet apt home by the park     2787   
1  2595                             Skylit Midtown Castle     2845   
2  3647               THE VILLAGE OF HARLEM....NEW YORK !     4632   
3  3831                   Cozy Entire Floor of Brownstone     4869   
4  5022  Entire Apt: Spacious Studio/Loft by central park     7192   

     host_name neighbourhood_group neighbourhood  latitude  longitude  \
0         John            Brooklyn    Kensington  40.64749  -73.97237   
1     Jennifer           Manhattan       Midtown  40.75362  -73.98377   
2    Elisabeth           Manhattan        Harlem  40.80902  -73.94190   
3  LisaRoxanne            Brooklyn  Clinton Hill  40.68514  -73.95976   
4        Laura           Manhattan   East Harlem  40.79851  -73.94399   

         room_type  price  minimum_nights  number_of_reviews last_review  \
0     Private room    149               1    

In [5]:
memory_before = df.memory_usage(deep=True).sum()

print("Memory usage before optimisation:")
print(f"{memory_before / 1024**2:.2f} MB")

Memory usage before optimisation:
19.16 MB


In [6]:
# Downcast integer columns
integer_columns = df.select_dtypes(include=["int64"]).columns

for col in integer_columns:
    df[col] = pd.to_numeric(df[col], downcast="integer")

# Downcast floating-point columns
float_columns = df.select_dtypes(include=["float64"]).columns

for col in float_columns:
    df[col] = pd.to_numeric(df[col], downcast="float")

# Convert low-cardinality text columns to category
category_columns = [
    "neighbourhood_group",
    "neighbourhood",
    "room_type"
]

for col in category_columns:
    df[col] = df[col].astype("category")

In [7]:
memory_after = df.memory_usage(deep=True).sum()

saving_percent = (
    (memory_before - memory_after) / memory_before
) * 100

print("Memory usage after optimisation:")
print(f"{memory_after / 1024**2:.2f} MB")

print(f"Memory saving: {saving_percent:.2f}%")

Memory usage after optimisation:
8.62 MB
Memory saving: 55.03%


In [8]:
profile = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": df.isna().sum().values,
    "missing_percentage": (
        df.isna().mean().values * 100
    ),
    "unique_count": df.nunique(dropna=True).values,
    "most_frequent": [
        df[col].mode(dropna=True).iloc[0]
        if not df[col].mode(dropna=True).empty
        else np.nan
        for col in df.columns
    ],
    "most_frequent_frequency": [
        df[col].value_counts(dropna=True).iloc[0]
        if not df[col].value_counts(dropna=True).empty
        else 0
        for col in df.columns
    ]
})

profile

,column,dtype,missing_count,missing_percentage,unique_count,most_frequent,most_frequent_frequency
0,id,int32,0,0.000000,48895,2539,1
1,name,string,16,0.032723,47905,Hillside Hotel,18
2,host_id,int32,0,0.000000,37457,219517861,327
3,host_name,string,21,0.042949,11452,Michael,417
4,neighbourhood_group,category,0,0.000000,5,Manhattan,21661
5,neighbourhood,category,0,0.000000,221,Williamsburg,3920
6,latitude,float32,0,0.000000,19048,40.718128,18
7,longitude,float32,0,0.000000,14718,-73.956772,18
8,room_type,category,0,0.000000,3,Entire home/apt,25409
9,price,int16,0,0.000000,674,100,2051


In [9]:
profile["missing_percentage"] = profile["missing_percentage"].round(2)

print("Complete DataFrame Profile:")
display(profile)

Complete DataFrame Profile:


,column,dtype,missing_count,missing_percentage,unique_count,most_frequent,most_frequent_frequency
0,id,int32,0,0.00,48895,2539,1
1,name,string,16,0.03,47905,Hillside Hotel,18
2,host_id,int32,0,0.00,37457,219517861,327
3,host_name,string,21,0.04,11452,Michael,417
4,neighbourhood_group,category,0,0.00,5,Manhattan,21661
5,neighbourhood,category,0,0.00,221,Williamsburg,3920
6,latitude,float32,0,0.00,19048,40.718128,18
7,longitude,float32,0,0.00,14718,-73.956772,18
8,room_type,category,0,0.00,3,Entire home/apt,25409
9,price,int16,0,0.00,674,100,2051


In [10]:
df_clean = df.copy()

In [11]:
# Fill numeric missing values with median
df_clean["reviews_per_month"] = (
    df_clean["reviews_per_month"]
    .fillna(df_clean["reviews_per_month"].median())
)

# Fill text missing values with mode
df_clean["name"] = (
    df_clean["name"]
    .fillna(df_clean["name"].mode()[0])
)

df_clean["host_name"] = (
    df_clean["host_name"]
    .fillna(df_clean["host_name"].mode()[0])
)

# Fill missing dates using the median date
median_date = df_clean["last_review"].dropna().sort_values().iloc[
    len(df_clean["last_review"].dropna()) // 2
]

df_clean["last_review"] = df_clean["last_review"].fillna(median_date)

In [12]:
missing_after = df_clean.isna().sum()

print("Missing values after cleaning:")
print(missing_after)

print("\nTotal remaining missing values:", missing_after.sum())

Missing values after cleaning:
id                                0
name                              0
host_id                           0
host_name                         0
neighbourhood_group               0
neighbourhood                     0
latitude                          0
longitude                         0
room_type                         0
price                             0
minimum_nights                    0
number_of_reviews                 0
last_review                       0
reviews_per_month                 0
calculated_host_listings_count    0
availability_365                  0
dtype: int64

Total remaining missing values: 0


In [13]:
duplicate_count = df_clean.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


In [14]:
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", df_clean.shape)

Shape after removing duplicates: (48895, 16)


In [15]:
print("Remaining duplicate rows:", df_clean.duplicated().sum())

Remaining duplicate rows: 0


In [16]:
# Clean text columns by removing leading/trailing whitespace
text_columns = [
    "name",
    "host_name",
    "neighbourhood_group",
    "neighbourhood",
    "room_type"
]

for col in text_columns:
    df_clean[col] = df_clean[col].str.strip()

In [17]:
# Standardise selected text fields
df_clean["neighbourhood_group"] = (
    df_clean["neighbourhood_group"].str.title()
)

df_clean["neighbourhood"] = (
    df_clean["neighbourhood"].str.title()
)

df_clean["room_type"] = (
    df_clean["room_type"].str.title()
)

In [18]:
print("Neighbourhood groups:")
print(df_clean["neighbourhood_group"].unique())

print("\nRoom types:")
print(df_clean["room_type"].unique())

Neighbourhood groups:
['Brooklyn' 'Manhattan' 'Queens' 'Staten Island' 'Bronx']

Room types:
['Private Room' 'Entire Home/Apt' 'Shared Room']


In [19]:
df_clean["last_review"] = pd.to_datetime(
    df_clean["last_review"],
    errors="coerce"
)

failed_dates = df_clean["last_review"].isna().sum()

print("Number of values that failed date parsing:", failed_dates)

df_clean["review_year"] = df_clean["last_review"].dt.year
df_clean["review_month"] = df_clean["last_review"].dt.month
df_clean["review_day_of_week"] = df_clean["last_review"].dt.day_name()
df_clean["review_quarter"] = df_clean["last_review"].dt.quarter

print("\nDate features created successfully.")

display(
    df_clean[
        [
            "last_review",
            "review_year",
            "review_month",
            "review_day_of_week",
            "review_quarter"
        ]
    ].head(10)
)

Number of values that failed date parsing: 0

Date features created successfully.


,last_review,review_year,review_month,review_day_of_week,review_quarter
0,2018-10-19,2018,10,Friday,4
1,2019-05-21,2019,5,Tuesday,2
2,2019-05-19,2019,5,Sunday,2
3,2019-07-05,2019,7,Friday,3
4,2018-11-19,2018,11,Monday,4
5,2019-06-22,2019,6,Saturday,2
6,2017-10-05,2017,10,Thursday,4
7,2019-06-24,2019,6,Monday,2
8,2017-07-21,2017,7,Friday,3
9,2019-06-09,2019,6,Sunday,2


In [20]:
def price_range(series):
    return series.max() - series.min()

group_summary = df_clean.groupby(
    "neighbourhood_group",
    observed=True
).agg(
    listing_count=("id", "count"),
    average_price=("price", "mean"),
    median_price=("price", "median"),
    average_reviews=("number_of_reviews", "mean"),
    price_range=("price", price_range)
).reset_index()

group_summary

,neighbourhood_group,listing_count,average_price,median_price,average_reviews,price_range
0,Bronx,1091,87.496792,65.0,26.004583,2500
1,Brooklyn,20104,124.383207,90.0,24.202845,10000
2,Manhattan,21661,196.875814,150.0,20.985596,10000
3,Queens,5666,99.517649,75.0,27.700318,9990
4,Staten Island,373,114.812332,75.0,30.941019,4987


In [21]:
group_avg_price = df_clean.groupby(
    "neighbourhood_group",
    observed=True
)["price"].transform("mean")

df_clean["group_avg_price"] = group_avg_price

df_clean["price_vs_group_average"] = (
    df_clean["price"] - df_clean["group_avg_price"]
)

display(
    df_clean[
        [
            "neighbourhood_group",
            "price",
            "group_avg_price",
            "price_vs_group_average"
        ]
    ].head(10)
)

,neighbourhood_group,price,group_avg_price,price_vs_group_average
0,Brooklyn,149,124.383207,24.616793
1,Manhattan,225,196.875814,28.124186
2,Manhattan,150,196.875814,-46.875814
3,Brooklyn,89,124.383207,-35.383207
4,Manhattan,80,196.875814,-116.875814
5,Manhattan,200,196.875814,3.124186
6,Brooklyn,60,124.383207,-64.383207
7,Manhattan,79,196.875814,-117.875814
8,Manhattan,79,196.875814,-117.875814
9,Manhattan,150,196.875814,-46.875814


In [22]:
def price_category(row):
    if row["price"] > row["group_avg_price"]:
        return "Above Average"
    elif row["price"] < row["group_avg_price"]:
        return "Below Average"
    else:
        return "Equal to Average"

df_clean["price_category"] = df_clean.apply(
    price_category,
    axis=1
)

display(
    df_clean[
        [
            "neighbourhood_group",
            "price",
            "group_avg_price",
            "price_category"
        ]
    ].head(10)
)

,neighbourhood_group,price,group_avg_price,price_category
0,Brooklyn,149,124.383207,Above Average
1,Manhattan,225,196.875814,Above Average
2,Manhattan,150,196.875814,Below Average
3,Brooklyn,89,124.383207,Below Average
4,Manhattan,80,196.875814,Below Average
5,Manhattan,200,196.875814,Above Average
6,Brooklyn,60,124.383207,Below Average
7,Manhattan,79,196.875814,Below Average
8,Manhattan,79,196.875814,Below Average
9,Manhattan,150,196.875814,Below Average


In [23]:
df_clean["group_total_reviews"] = df_clean.groupby(
    "neighbourhood_group",
    observed=True
)["number_of_reviews"].transform("sum")

df_clean["review_percentage_in_group"] = (
    df_clean["number_of_reviews"]
    / df_clean["group_total_reviews"]
) * 100

display(
    df_clean[
        [
            "neighbourhood_group",
            "number_of_reviews",
            "group_total_reviews",
            "review_percentage_in_group"
        ]
    ].head(10)
)

,neighbourhood_group,number_of_reviews,group_total_reviews,review_percentage_in_group
0,Brooklyn,9,486574,0.001850
1,Manhattan,45,454569,0.009899
2,Manhattan,0,454569,0.000000
3,Brooklyn,270,486574,0.055490
4,Manhattan,9,454569,0.001980
5,Manhattan,74,454569,0.016279
6,Brooklyn,49,486574,0.010070
7,Manhattan,430,454569,0.094595
8,Manhattan,118,454569,0.025959
9,Manhattan,160,454569,0.035198
